# LookBackLens baseline — Hallucination Detection in Tool Calling

Implements the LookBackLens method (Chuang et al., 2024) using `meta-llama/Llama-3.2-3B-Instruct` as the backbone (~3.2B params, fits on a Kaggle P100/T4 in fp16).

Pipeline:
1. For each train sample, do one forward pass with output_attentions=True. Extract per-token lookback ratios over the answer span.
2. For each hallucinated train sample, the gold span is the **positive** training example. Add `K=3` random non-overlapping windows of similar size as **negative** training examples. For clean train samples, only negatives.
3. Train a logistic regression on these `L*H`-dim feature vectors.
4. For test, do one forward pass per sample, slide a fixed-size window across the answer tokens, classify each window. Merge consecutive positive windows into char spans.
5. Evaluate against gold using the same span-overlap metrics we use for LettuceDetect / LLM detector.

Upload `combined_train.jsonl`, `combined_val.jsonl`, `combined_test.jsonl` to `/content/` (Colab) or as a Kaggle dataset under `/kaggle/input/halluc-toolace/`.

Requires GPU and a HF token with access to Llama-3.2 (add as Kaggle / Colab secret `HF_TOKEN`).

In [ ]:
import subprocess, sys, os, importlib.util

NEEDED = ("transformers", "accelerate", "sklearn")
missing = [p for p in NEEDED if importlib.util.find_spec(p) is None]
if missing:
    pkgs = []
    for p in missing:
        pkgs.append("scikit-learn" if p == "sklearn" else p)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers>=4.48", "accelerate", "scikit-learn"])
    if "google.colab" in sys.modules:
        print("Restarting kernel to load new packages…")
        os.kill(os.getpid(), 9)
else:
    print("All packages already installed.")


In [ ]:
from pathlib import Path

CANDIDATE_DIRS = [
    Path("/content"),
    Path("/kaggle/input/halluc-toolace"),
    Path("data"),
    Path("."),
]
REQUIRED = ["combined_train.jsonl", "combined_val.jsonl", "combined_test.jsonl"]
DATA_DIR = next((d for d in CANDIDATE_DIRS if all((d / f).exists() for f in REQUIRED)), None)

if DATA_DIR is None:
    print("Data not found. Upload via the next cell or set DATA_DIR manually.")
else:
    print(f"Using DATA_DIR = {DATA_DIR}")

# Backbone LM that exposes per-layer attention. Qwen2.5-3B is fully open (no HF auth).
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MAX_LENGTH = 4096
WINDOW = 8                  # chunk size from the paper (Table 3)
STRIDE = 8                  # non-overlapping chunks (paper's sliding-window setup)
THRESHOLD = 0.5
SEED = 42

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"CUDA: {torch.cuda.is_available()}",
      f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")


In [ ]:
if DATA_DIR is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_DIR = Path("/content")
    except ImportError:
        raise RuntimeError("Set DATA_DIR manually if not in Colab.")
for f in REQUIRED:
    print(f"  {f}: {(DATA_DIR / f).exists()}")


## Helpers (data, extraction, metrics)

All code is inlined so the notebook is self-contained.

In [ ]:
import json
from dataclasses import dataclass

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

# ----- metrics -----
def _char_set(spans):
    out = set()
    for s in spans:
        out.update(range(int(s["start"]), int(s["end"])))
    return out

def span_micro_f1(samples, pred_spans):
    o = p = g = 0
    for s, ps in zip(samples, pred_spans):
        gs = _char_set(s["hallucination_labels"]); pset = _char_set(ps)
        o += len(gs & pset); p += len(pset); g += len(gs)
    pr = o/p if p else 0.0; re = o/g if g else 0.0
    f1 = 2*pr*re/(pr+re) if (pr+re) else 0.0
    return pr, re, f1

def span_macro_f1(samples, pred_spans):
    fs = []
    for s, ps in zip(samples, pred_spans):
        gs = _char_set(s["hallucination_labels"]); pset = _char_set(ps)
        o = len(gs & pset)
        if not pset and not gs:
            fs.append(1.0); continue
        pr = o/len(pset) if pset else 0.0
        re = o/len(gs) if gs else 0.0
        fs.append(2*pr*re/(pr+re) if (pr+re) else 0.0)
    return sum(fs)/len(fs) if fs else 0.0

def example_f1(samples, pred_spans):
    tp = fp = fn = 0
    for s, ps in zip(samples, pred_spans):
        gold = bool(s["hallucination_labels"]); pred = bool(ps)
        if gold and pred: tp += 1
        elif gold: fn += 1
        elif pred: fp += 1
    pr = tp/(tp+fp) if (tp+fp) else 0.0
    re = tp/(tp+fn) if (tp+fn) else 0.0
    f1 = 2*pr*re/(pr+re) if (pr+re) else 0.0
    return pr, re, f1

# ----- LookBackLens core: forward pass + per-token lookback ratios -----
import numpy as np
import torch

@dataclass
class ExtractionOut:
    ratios: np.ndarray
    answer_token_offsets: np.ndarray
    answer_text: str

@torch.no_grad()
def extract_lookback_ratios(model, tokenizer, sample, device="cuda", max_length=MAX_LENGTH):
    context_block = f"{sample['context']}\n\nQuestion: {sample['query']}\n\nAnswer: "
    enc_ctx = tokenizer(context_block, add_special_tokens=True, return_tensors="pt")
    enc_ans = tokenizer(sample["output"], add_special_tokens=False, return_tensors="pt",
                        return_offsets_mapping=True)
    input_ids = torch.cat([enc_ctx["input_ids"], enc_ans["input_ids"]], dim=1)
    if input_ids.size(1) > max_length:
        excess = input_ids.size(1) - max_length
        ctx_len = enc_ctx["input_ids"].size(1)
        keep = max(0, ctx_len - excess)
        input_ids = torch.cat([enc_ctx["input_ids"][:, :keep], enc_ans["input_ids"]], dim=1)
    attention_mask = torch.ones_like(input_ids)
    context_end = input_ids.size(1) - enc_ans["input_ids"].size(1)
    num_ans = enc_ans["input_ids"].size(1)

    input_ids = input_ids.to(device); attention_mask = attention_mask.to(device)
    out = model(input_ids=input_ids, attention_mask=attention_mask,
                output_attentions=True, use_cache=False)
    L = len(out.attentions); H = out.attentions[0].size(1)

    tril = torch.tril(torch.ones(num_ans, num_ans, device=device, dtype=torch.float16), diagonal=-1)
    counts = torch.arange(num_ans, device=device, dtype=torch.float16).clamp(min=1)

    ratios = torch.empty(L, H, num_ans, device=device, dtype=torch.float16)
    for l, attn in enumerate(out.attentions):
        a = attn[0].to(torch.float16)
        ans_rows = a[:, context_end:context_end + num_ans, :]
        att_ctx = ans_rows[:, :, :context_end].mean(dim=-1)
        a2a = ans_rows[:, :, context_end:context_end + num_ans]
        att_new = (a2a * tril).sum(dim=-1) / counts
        att_new[:, 0] = 0
        ratios[l] = att_ctx / (att_ctx + att_new + 1e-6)

    return ExtractionOut(
        ratios=ratios.cpu().numpy(),
        answer_token_offsets=enc_ans["offset_mapping"][0].numpy(),
        answer_text=sample["output"],
    )

def _tok_overlap(offsets, s, e):
    return [i for i, (a, b) in enumerate(offsets) if a < e and b > s and a != b]

def span_feature(ratios, idxs):
    if not idxs:
        return np.zeros(ratios.shape[0] * ratios.shape[1], dtype=np.float16)
    return ratios[:, :, idxs].mean(axis=-1).reshape(-1).astype(np.float16)

def sliding_windows(ratios, window, stride):
    n = ratios.shape[-1]
    if n == 0: return np.zeros((0, ratios.shape[0]*ratios.shape[1]), dtype=np.float16), []
    if n < window: window = n
    feats, idx_lists = [], []
    for s in range(0, n - window + 1, stride):
        idxs = list(range(s, s + window))
        feats.append(span_feature(ratios, idxs)); idx_lists.append(idxs)
    return np.stack(feats), idx_lists

def windows_to_spans(offsets, idx_lists, scores, threshold):
    """Paper-style: merge consecutive positive chunks into one char span."""
    if not idx_lists: return []
    spans, cs, ce, cm = [], None, None, 0.0
    for idxs, sc in zip(idx_lists, scores):
        if sc > threshold:
            chunk_start = int(offsets[idxs[0], 0])
            chunk_end = int(offsets[idxs[-1], 1])
            if cs is None:
                cs, ce = chunk_start, chunk_end
            elif chunk_start <= ce:
                ce = max(ce, chunk_end)
            else:
                spans.append({"start": cs, "end": ce, "confidence": cm})
                cs, ce, cm = chunk_start, chunk_end, 0.0
            cm = max(cm, float(sc))
        else:
            if cs is not None:
                spans.append({"start": cs, "end": ce, "confidence": cm})
                cs = ce = None; cm = 0.0
    if cs is not None:
        spans.append({"start": cs, "end": ce, "confidence": cm})
    return spans


## Load model + tokenizer

If you don't have a HF token with access to Llama-3.2, swap `BASE_MODEL` to a fully-open alternative like `Qwen/Qwen2.5-3B-Instruct` or `google/gemma-2-2b-it`.

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Pick up HF token from Colab/Kaggle secrets if present
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token: os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token: os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    attn_implementation="eager",   # eager attention so output_attentions=True works
    device_map=device,
)
model.eval()
print(f"Loaded {BASE_MODEL} on {device}; params={sum(p.numel() for p in model.parameters())/1e9:.2f}B")


## Extract features

For TRAIN: one positive feature vector per hallucinated span + up to `N_NEG_PER_SAMPLE` random negative windows of similar size. For clean train samples, only negatives.

For VAL/TEST: only sliding-window features (used both for tuning threshold on val and for final predictions).

~2-3 hours on Kaggle P100 / T4 for train + val + test. Progress is saved to `/content/lookback_progress.jsonl` so reruns resume.

In [ ]:
from tqdm.auto import tqdm

train_samples = read_jsonl(DATA_DIR / "combined_train.jsonl")
val_samples   = read_jsonl(DATA_DIR / "combined_val.jsonl")
test_samples  = read_jsonl(DATA_DIR / "combined_test.jsonl")
print(f"train={len(train_samples)}, val={len(val_samples)}, test={len(test_samples)}")

# TRAIN: sliding window over every answer; label each window 1 if it overlaps ANY gold span, 0 otherwise.
train_X, train_y = [], []
for s in tqdm(train_samples, desc="train extract"):
    out = extract_lookback_ratios(model, tokenizer, s, device=device)
    feats, idx_lists = sliding_windows(out.ratios, WINDOW, STRIDE)
    # gold token indices for this sample
    gold_idxs = set()
    for span in s["hallucination_labels"]:
        gold_idxs.update(_tok_overlap(out.answer_token_offsets, int(span["start"]), int(span["end"])))
    for feat, idxs in zip(feats, idx_lists):
        train_X.append(feat)
        train_y.append(1 if any(i in gold_idxs for i in idxs) else 0)

train_X = np.stack(train_X, axis=0).astype(np.float32)
train_y = np.array(train_y, dtype=np.int64)
print(f"\nTRAIN windows: {train_X.shape}  pos={int((train_y==1).sum())} neg={int((train_y==0).sum())}")


## Fit linear classifier (logistic regression)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000, solver="lbfgs", random_state=SEED)
clf.fit(train_X, train_y)
print(f"Train accuracy: {clf.score(train_X, train_y):.3f}")


## Test inference (sliding window) + metrics

In [ ]:
import pandas as pd

def predict_spans(s):
    out = extract_lookback_ratios(model, tokenizer, s, device=device)
    feats, idx_lists = sliding_windows(out.ratios, WINDOW, STRIDE)
    if feats.shape[0] == 0:
        return []
    scores = clf.predict_proba(feats.astype(np.float32))[:, 1]
    return windows_to_spans(out.answer_token_offsets, idx_lists, scores, THRESHOLD)

test_preds = [predict_spans(s) for s in tqdm(test_samples, desc="test inference")]

def filter_subset(label):
    sel_s, sel_p = [], []
    for s, p in zip(test_samples, test_preds):
        if not s["hallucination_labels"]:
            sel_s.append(s); sel_p.append(p); continue
        if s["hallucination_labels"][0].get("type") == label:
            sel_s.append(s); sel_p.append(p)
    return sel_s, sel_p

subsets = {
    "Combined":           (test_samples, test_preds),
    "Type 1 + clean":     filter_subset("Type1_Contradiction"),
    "Type 2 + clean":     filter_subset("Type2_Overgeneration"),
    "Type 3 + clean":     filter_subset("Type3_MissingTool"),
}
rows = []
for name, (s, p) in subsets.items():
    sp, sr, sf = span_micro_f1(s, p)
    macro = span_macro_f1(s, p)
    ep, er, ef = example_f1(s, p)
    rows.append({"Split": name, "N": len(s),
                 "Span P": round(sp, 3), "Span R": round(sr, 3), "Span F1": round(sf, 3),
                 "Span macro F1": round(macro, 3),
                 "Ex P": round(ep, 3), "Ex R": round(er, 3), "Ex F1": round(ef, 3)})
    print(f"{name} (N={len(s)}): span P/R/F1 = {sp:.3f}/{sr:.3f}/{sf:.3f} | macro F1 = {macro:.3f} "
          f"| ex P/R/F1 = {ep:.3f}/{er:.3f}/{ef:.3f}")
pd.DataFrame(rows)
